In [1]:
import sys

print(sys.executable)

/Users/juanestevez/Documents/Developer/Personal/ert-project/.venv/bin/python


In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ert_visualization.io import (
    read_electrode_elevations,
    read_ert_xyz,
)

In [3]:
# Find the repository root regardless of whether the notebook
# is launched from the project root or the notebooks directory.

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
FIGURE_DIR = PROJECT_ROOT / "figures"
INTERACTIVE_DIR = PROJECT_ROOT / "interactive"

FIGURE_DIR.mkdir(exist_ok=True)
INTERACTIVE_DIR.mkdir(exist_ok=True)

In [4]:
survey_files = {
    "Wenner Day 1": RAW_DATA_DIR / "Wenner_Day1.xyz",
    "Dipole-Dipole Day 1": RAW_DATA_DIR / "Dipole_Dipole_Day1.xyz",
    "Wenner Day 2": RAW_DATA_DIR / "Wenner_Day2.xyz",
    "Dipole-Dipole Day 2": RAW_DATA_DIR / "Dipole_Dipole_Day2.xyz",
}

ert_data = {
    survey_name: read_ert_xyz(filepath)
    for survey_name, filepath in survey_files.items()
}

In [5]:
for survey_name, df in ert_data.items():
    print(f"{survey_name}")
    print(f"  Model blocks: {len(df)}")
    print(
        f"  Profile distance: "
        f"{df['x_m'].min():.1f}–{df['x_m'].max():.1f} m"
    )
    print(
        f"  Maximum depth: "
        f"{df['depth_positive_m'].max():.1f} m"
    )
    print(
        f"  Resistivity: "
        f"{df['resistivity_ohm_m'].min():.2f}–"
        f"{df['resistivity_ohm_m'].max():.2f} Ω·m"
    )
    print()

Wenner Day 1
  Model blocks: 748
  Profile distance: 6.0–314.0 m
  Maximum depth: 52.5 m
  Resistivity: 5.45–35794.00 Ω·m

Dipole-Dipole Day 1
  Model blocks: 977
  Profile distance: 6.0–394.0 m
  Maximum depth: 45.9 m
  Resistivity: 0.62–1327537.00 Ω·m

Wenner Day 2
  Model blocks: 644
  Profile distance: 4.5–229.5 m
  Maximum depth: 29.9 m
  Resistivity: 1.96–156818.00 Ω·m

Dipole-Dipole Day 2
  Model blocks: 727
  Profile distance: 4.5–235.5 m
  Maximum depth: 34.4 m
  Resistivity: 0.55–293065.00 Ω·m



In [6]:
elevation_data = read_electrode_elevations(
    DATA_DIR / "electrode_elevations.csv"
)

day1_elevation = elevation_data[
    elevation_data["line"] == "Day1"
].copy()

day2_elevation = elevation_data[
    elevation_data["line"] == "Day2"
].copy()

In [7]:
for survey_name, df in ert_data.items():
    print(
        f"{survey_name}: "
        f"{df['x_m'].min():.1f} to {df['x_m'].max():.1f} m"
    )

print()
print(
    "Day 1 elevation data: "
    f"{day1_elevation['x_m'].min():.1f} to "
    f"{day1_elevation['x_m'].max():.1f} m"
)

print(
    "Day 2 elevation data: "
    f"{day2_elevation['x_m'].min():.1f} to "
    f"{day2_elevation['x_m'].max():.1f} m"
)

Wenner Day 1: 6.0 to 314.0 m
Dipole-Dipole Day 1: 6.0 to 394.0 m
Wenner Day 2: 4.5 to 229.5 m
Dipole-Dipole Day 2: 4.5 to 235.5 m

Day 1 elevation data: 0.0 to 320.0 m
Day 2 elevation data: 0.0 to 240.0 m


### Survey geometry note

Day 1 and Day 2 were collected along perpendicular profiles with different electrode spacing. The Day 1 elevation data span 0–320 m, while the Dipole-Dipole Day 1 inversion model extends to approximately 394 m. Because the horizontal coordinates do not directly match the available topographic profile, topographic correction is not applied to the Dipole-Dipole Day 1 section until the coordinate relationship is verified.

In [8]:
from ert_visualization.io import (
    read_electrode_elevations,
    read_ert_xyz,
)

from ert_visualization.processing import (
    add_topography_to_model,
    interpolate_ert_grid,
    validate_topography_coverage,
)

In [9]:
wenner_day1_topo = add_topography_to_model(
    ert_data["Wenner Day 1"],
    day1_elevation,
)

wenner_day1_topo[
    [
        "x_m",
        "depth_positive_m",
        "surface_elevation_m",
        "model_elevation_m",
    ]
].head()

,x_m,depth_positive_m,surface_elevation_m,model_elevation_m
0,6.0,1.0,1252.248820,1251.248820
1,10.0,1.0,1252.106825,1251.106825
2,14.0,1.0,1251.930560,1250.930560
3,18.0,1.0,1251.765620,1250.765620
4,22.0,1.0,1251.668640,1250.668640


In [10]:
wenner_day2_topo = add_topography_to_model(
    ert_data["Wenner Day 2"],
    day2_elevation,
)

dipole_day2_topo = add_topography_to_model(
    ert_data["Dipole-Dipole Day 2"],
    day2_elevation,
)

In [11]:
dipole_day1_topo = add_topography_to_model(
    ert_data["Dipole-Dipole Day 1"],
    day1_elevation,
)

ValueError: Topography data does not cover the ERT survey area. ERT model spans 6.0 - 394.0 m, but elevation data spans 0.0 - 320.0 m.

In [12]:
XI, ZI, RHOI = interpolate_ert_grid(
    ert_data["Wenner Day 1"]
)

In [13]:
print("XI shape:", XI.shape)
print("ZI shape:", ZI.shape)
print("RHOI shape:", RHOI.shape)

XI shape: (200, 400)
ZI shape: (200, 400)
RHOI shape: (200, 400)


In [14]:
XI_topo, ZI_topo, RHOI_topo = interpolate_ert_grid(
    wenner_day1_topo,
    vertical_column="model_elevation_m",
)

In [15]:
print(XI_topo.shape)
print(ZI_topo.shape)
print(RHOI_topo.shape)

(200, 400)
(200, 400)
(200, 400)
